# ANN
Simple neural network implementation.

Lets assume a two-layer network with 10 hidden units in the hidden layer and 1 output units for regression.

$$
\begin{align*}
\text{Input layer} & \rightarrow \text{Hidden layer} & \rightarrow \text{Output layer} \\
\mathbf{x} & \rightarrow \mathbf{h} & \rightarrow \hat{y} \\
\mathbf{h} & = \sigma(\mathbf{x} \mathbf{W}_1 + \mathbf{b}_1) & \hat{y} = \mathbf{h} \mathbf{W}_2 + \mathbf{b}_2
\end{align*}
$$

Input/output shapes are as follows:
- $\mathbf{x}$: (N, d) input feature vector
- $\mathbf{a}$: (N, 10) hidden layer activations
- $\hat{y}$: (N, 1) output predictions

Parameters:
- $\mathbf{W}_1$: (d, 10) weights for hidden layer
- $\mathbf{b}_1$: (10,) biases for hidden layer
- $\mathbf{W}_2$: (10, 1) weights for output layer
- $\mathbf{b}_2$: (1,) biases for output layer

## Forward Pass

As illustrated above, the forward pass for such a network is:
- $\mathbf{a} = \sigma (\mathbf{h}) = \sigma(\mathbf{x} \mathbf{W}_1 + \mathbf{b}_1)$
- $\hat{y} = \mathbf{a} \mathbf{W}_2 + \mathbf{b}_2$

$\sigma$ is an activation function, such as ReLU or sigmoid. 
For simplicity, we can use ReLU: $\sigma(z) = \max(0, z)$.


## Backward Pass
Bakkward pass is to compute the gradients of the output in terms of the parameters, which is needed for optimization/training.

Assume the gradient of the loss with respect to the output is $G = \frac{\partial L}{\partial \hat{y}} \rightarrow (N, )$, then we can compute the gradients as follows:

$$
\begin{align*}
\frac{\partial \mathbf{L}}{\partial \mathbf{W}_2} & = \frac{\partial \mathbf{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial \mathbf{W}_2} = \mathbf{a}^T G \\
\frac{\partial \mathbf{L}}{\partial \mathbf{b}_2} & = 1|_{(1, N)} G \\
\frac{\partial \mathbf{L}}{\partial \mathbf{a}} & = G\mathbf{W}_2^T \\
\frac{\partial \mathbf{L}}{\partial \mathbf{W}_1} & = \frac{\partial \mathbf{L}}{\partial \mathbf{a}} \cdot \frac{\partial \mathbf{a}}{\partial \mathbf{h}} \cdot \frac{\partial \mathbf{h}}{\partial \mathbf{W}_1} = \mathbf{x}^T \cdot (\frac{\partial \mathbf{L}}{\partial \mathbf{a}} \odot \frac{\partial \mathbf{a}}{\partial \mathbf{h}}) \\
\frac{\partial \mathbf{L}}{\partial \mathbf{b}_1} & = \sum_{i=1}^N (\frac{\partial \mathbf{L}}{\partial \mathbf{a}} \odot \frac{\partial \mathbf{a}}{\partial \mathbf{h}}) \\
\frac{\partial \mathbf{a}}{\partial \mathbf{h}} &= \begin{cases}
1 & \text{if } h > 0 \\
0 & \text{otherwise}
\end{cases}
\end{align*}
$$

$\cdot$ denotes maxtrix multiplication, $\odot$ denotes element-wise multiplication, and $1|_{(1, N)}$ denotes a row vector of ones with shape (1, N) for summing over the batch dimension.

## Implementation using Numpy

In [5]:
import numpy as np 

class NeuralNetwork:
    def __init__(self, in_features, hidden_units, out_features):
        # Initialize weights and biases
        self.W1 = np.random.randn(in_features, hidden_units) * 0.01
        self.b1 = np.zeros((hidden_units,))
        self.W2 = np.random.randn(hidden_units, out_features) * 0.01
        self.b2 = np.zeros((out_features,))
    
    
    def forward(self, x):
        """Forward pass through the network

        Args:
            X (np.ndarray): Input data of shape (batch_size, in_features)
        """
        h = np.dot(x, self.W1) + self.b1  # Linear transformation for hidden layer
        a = np.maximum(0, h)  # ReLU activation
        out = np.dot(a, self.W2) + self.b2  # Linear transformation for output layer
        return out, (h, a)  # Return intermediate values for backward pass
    
    
    def backward(self, x, h, a, out, G):
        """Backward pass to compute gradients

        Args:
            x (np.ndarray): Input data of shape (batch_size, in_features)
            h (np.ndarray): Hidden layer activations of shape (batch_size, hidden_units)
            out (np.ndarray): Output predictions of shape (batch_size, out_features)
            G (np.ndarray): Gradient of loss w.r.t. output of shape (batch_size, out_features)
        
        Returns:
            dW1 (np.ndarray): Gradient of loss w.r.t. W1
            db1 (np.ndarray): Gradient of loss w.r.t. b1
            dW2 (np.ndarray): Gradient of loss w.r.t. W2, shape (hidden_units, out_features)
            db2 (np.ndarray): Gradient of loss w.r.t. b2
        """
        B = x.shape[0]  # batch size
        dw2 = a.T @ G
        db2 = np.ones((1, B)) @ G
        
        da = G @ self.W2.T
        dadh = (h > 0).astype(float)  # ReLU derivative
        
        dw1 = x.T @ (da * dadh)
        db1 = np.ones((1, B)) @ (da * dadh)
        
        return dw1, db1, dw2, db2

In [8]:
# some data 
samples = 500
in_features = 20
hidden_units = 10
out_features = 1

# training settings
max_iters = 2000
learning_rate = 0.01


nn = NeuralNetwork(in_features, hidden_units, out_features)

# main train loop
for epoch in range(max_iters):
    # forward pass
    out, (h, a) = nn.forward(X)
    
    # loss function (MSE)
    loss = np.mean((out - y) ** 2)
    
    # Compute gradient of loss w.r.t. output
    G = (2 / samples) * (out - y)
    
    dW1, db1, dW2, db2 = nn.backward(X, h, a, out, G)
    
    # Update parameters
    nn.W1 -= learning_rate * dW1
    nn.b1 -= learning_rate * db1.flatten()
    nn.W2 -= learning_rate * dW2
    nn.b2 -= learning_rate * db2.flatten()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")



Epoch 0, Loss: 0.9870
Epoch 100, Loss: 0.9835
Epoch 200, Loss: 0.9832
Epoch 300, Loss: 0.9830
Epoch 400, Loss: 0.9825
Epoch 500, Loss: 0.9817
Epoch 600, Loss: 0.9805
Epoch 700, Loss: 0.9784
Epoch 800, Loss: 0.9749
Epoch 900, Loss: 0.9696
Epoch 1000, Loss: 0.9618
Epoch 1100, Loss: 0.9513
Epoch 1200, Loss: 0.9389
Epoch 1300, Loss: 0.9263
Epoch 1400, Loss: 0.9147
Epoch 1500, Loss: 0.9045
Epoch 1600, Loss: 0.8956
Epoch 1700, Loss: 0.8858
Epoch 1800, Loss: 0.8762
Epoch 1900, Loss: 0.8665
